In [5]:
from torch.utils.data import TensorDataset
import numpy as np
import torch
from tqdm import tqdm

from SpringSim import SpringSim
from nri.ElboObjective import ElboLoss
from nri.NRI import NRIModule

In [2]:
prior = np.array([0.5, 0, 0.5])

def create_dataset(size: int):
    sim = SpringSim()
    trajectories = []
    for _ in tqdm(range(size)):
        xs, vs, edge = sim.sample_trajectory(1000, 10, prior)
        node_features = np.concatenate([xs, vs], axis=1)
        trajectories.append(node_features.transpose(0, 2, 1))

    return TensorDataset(torch.Tensor(np.stack(trajectories, axis=0)))

ds_train = create_dataset(1000)
ds_test = create_dataset(100)

100%|██████████| 100/100 [00:02<00:00, 49.12it/s]


In [7]:
from torch.utils.tensorboard import SummaryWriter
from nri.train_nri import train

summary_writer = SummaryWriter("../data/logs/test_nri")
nri_module = NRIModule(
    x_dim=4,
    hidden_dim=8,
    trajectory_length=99,
    num_edge_types=3,
    pred_steps=4,
    dropout_prob=0.1,
    skip_first=True
)
criterion = ElboLoss(prior=prior)
train(nri_module, ds_train, ds_test, criterion=criterion, tensorboard_logger=summary_writer, num_epochs=50)

Testing: 100%|██████████| 2/2 [00:00<00:00, 16.80it/s]


NRIModule(
  (encoder): Encoder(
    (f_emb): MLP(
      (fc1): Linear(in_features=396, out_features=8, bias=True)
      (fc2): Linear(in_features=8, out_features=8, bias=True)
      (bn): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (node2edge_1): Node2Edge(
      (psi): MLP(
        (fc1): Linear(in_features=16, out_features=8, bias=True)
        (fc2): Linear(in_features=8, out_features=8, bias=True)
        (bn): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (edge2node): Edge2Node(
      (phi): MLP(
        (fc1): Linear(in_features=8, out_features=8, bias=True)
        (fc2): Linear(in_features=8, out_features=8, bias=True)
        (bn): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (node2edge_2): Node2Edge(
      (psi): MLP(
        (fc1): Linear(in_features=16, out_features=8, bias=True)
        (fc2): Linear(in_features=8, out_feature